[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/01-prerequisites/ml-prereq-inferential-stats.ipynb)

# Inferential Statistics

*AIBits Academy · Machine Learning End To End · Prerequisites*

Hypothesis testing, confidence intervals, and the tests (t, F, chi-square, correlation) that reappear throughout Evaluation & Validation and every Full Project's "is this real or noise?" question.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

## The Central Limit Theorem

The Central Limit Theorem (CLT) states that for a sufficiently large sample size, the sampling distribution of the sample mean approaches a normal distribution — regardless of the shape of the underlying population. This is the reason z-tests and t-tests are allowed to assume normality even when the raw data isn't normally distributed: it's the *sampling distribution of the mean* that becomes normal, not the raw data itself.

### Seeing the CLT Happen: A Simulation

That claim is easy to state and easy to doubt — so here's what it looks like when you actually run it. Take a fair six-sided die (population mean μ = 3.5, values 1–6 — about as far from "bell-shaped" as a distribution gets, since every value is equally likely). Roll it repeatedly, average groups of *n* rolls together, and plot the distribution of those averages:

In [ ]:
import numpy as np

die_values = [1, 2, 3, 4, 5, 6]
for sample_size in [2, 4, 8, 16, 32]:
    sample_means = [np.mean(np.random.choice(die_values, size=sample_size)) for _ in range(1000)]
    # fit & plot a normal curve over these 1,000 sample means

The result: at sample size *n*=2, the distribution of sample means is still visibly wide and blocky. By *n*=32, it has tightened into a smooth, distinctly bell-shaped curve centered exactly on 3.5 — even though not a single individual die roll is bell-shaped at all. A separate run, drawing directly from a simulated population of one million die rolls (confirmed population mean: exactly 3.5), tracked how many samples of each size it took for the running mean of sample means to converge to 3.50:

| Sample size (n) | Samples needed to converge to running mean = 3.50 |
|---|---|
| 2 | 318 |
| 4 | 350 |
| 8 | 147 |
| 16 | 61 |
| 32 | 20 |
| 64 | 22 |

Separately, holding sample size fixed at n=32 and instead increasing the *number* of samples averaged together shows the same convergence from a different angle — the running mean of sample means starts at 3.88 after just 2 samples, and settles to 3.50 by around 1,000 samples, staying there through 100,000 and 1,000,000 samples. This is the Law of Large Numbers and the CLT working together: more samples make the average of sample means converge to the true population mean, while a larger individual sample size makes each sample mean's own distribution more normal-shaped.

## Interactive: Roll the Dice and Watch the CLT Happen

Pick a sample size, then roll. Each click draws that many die values, averages them, and adds one more bar to the histogram of sample means below — watch a flat, uniform die distribution turn into a bell curve purely through repeated averaging by repeatedly clicking the **Roll ×1** button. Then try the same with the **Roll ×50** button, and notice how much faster and tighter the bell curve forms when each sample mean is already an average of 50 dice.

## Parameter Estimation: Method of Moments & Maximum Likelihood

Every test on this page assumes you already have an estimate of a population parameter (a mean, a proportion, a variance) to test — but where does that estimate actually come from? Two systematic approaches show up throughout this course, usually without being named: whenever Logistic Regression or a GLM is "fit," what's actually happening under the hood is Maximum Likelihood Estimation.

### Method of Moments

The oldest and simplest approach: set the sample's moments (mean, variance, ...) equal to the distribution's theoretical moments (which are functions of the unknown parameters), then solve for those parameters.

> **💡 Worked example**
>
> For a Uniform distribution on [0, θ], the theoretical mean is θ/2. If a sample of delivery times from a Zomato zone has a sample mean of 18 minutes, the method-of-moments estimate is immediate: set θ/2 = 18, giving θ̂ = 36 minutes as the estimated maximum possible delivery time.

### Maximum Likelihood Estimation (MLE)

A more general — and almost always more efficient — approach: choose the parameter value that makes the *observed data* most probable. Formally, find the parameter θ that maximises the likelihood function L(θ) = P(data | θ); in practice, the log-likelihood is maximised instead, since it turns a product of probabilities into a sum, which is far easier to differentiate.

$$\hat{\theta}_{\text{MLE}} = \operatorname*{argmax}_{\theta}\ \sum \log P(x_i\mid\theta)$$

**The core intuition.** Flip the usual question around. Normally probability asks: "given a known parameter, how likely is this data?" MLE asks the reverse: "given this data I've actually observed, which parameter value would have made it most likely?" You try many candidate parameter values, score each one by how well it explains the data you're holding, and keep the best-scoring candidate. The word *likelihood* is deliberately not *probability*: probability is a function of the data with the parameter fixed; likelihood is a function of the parameter with the data fixed. Same formula, opposite thing held constant.

**Why the product, and why the log.** If the observations are independent, the probability of seeing the whole dataset is the product of each point's probability: L(θ) = P(x₁|θ) · P(x₂|θ) · … · P(xₙ|θ). Multiplying many small probabilities together produces a vanishingly tiny number that underflows to zero on a computer and is painful to differentiate. Taking the logarithm converts that product into a sum — log L(θ) = Σ log P(xᵢ|θ) — which is numerically stable and, because log is monotonically increasing, is maximised at exactly the same θ as the original likelihood. That is why virtually every ML library minimises a *negative log-likelihood* (equivalently, maximises log-likelihood) rather than the raw likelihood.

**A concrete case.** Suppose you measure the heights of 8 people and are willing to assume they come from a Normal distribution with a known spread σ. MLE for the mean μ asks: of all possible values for the true average height, which one makes these 8 specific measurements most probable? A μ far below the data assigns tiny probability to every tall observation; a μ far above does the same for the short ones; only a μ sitting in the middle of the data scores highly on all of them at once. It turns out — provably, by setting the derivative of the log-likelihood to zero — that the maximiser is simply the sample mean. MLE doesn't always give such a clean closed form (Logistic Regression's has none, which is why it's solved numerically by gradient descent), but the *principle* is identical every time.

### Interactive: Which Parameter Best Explains the Data?

The eight orange dots below are observed height measurements (cm). Each panel tests one candidate value for the true mean μ, drawing the Normal curve that value implies and reporting the resulting likelihood — how probable these exact eight points are under that curve. Watch the likelihood climb from near-zero (a badly-placed μ) to its maximum (the MLE, which lands right at the sample mean).

Orange dots = the 8 observed heights (on the axis); the small gold dot above each shows how much probability the candidate curve assigns to that point. A candidate μ scores well only when *all* eight points sit under the high-density part of its curve — which happens exactly at the sample mean, μ = 169.5 cm. That's why the MLE for a Normal mean is provably just the average of the data.

This is precisely the mechanism behind the **Logistic Regression** page's log-loss cost function: minimising log-loss is mathematically identical to maximising the log-likelihood of the observed classes under a Bernoulli model. Every GLM in this course (Logistic, Poisson, and beyond) is fit by this same principle — only the assumed distribution for the response variable changes.

## Hypothesis Testing — The Full Framework

Every statistical test in this section follows the same six-step skeleton:

1. **Define the null hypothesis (H₀)** — the default, "no effect" assumption. e.g. "Mean petrol price in Ahmedabad = ₹96/litre."
2. **Define the alternative hypothesis (Hₐ)** — what you're trying to show. e.g. "Mean petrol price ≠ ₹96/litre" (two-tailed) or "> ₹96" (one-tailed).
3. **Choose a significance level (α)** — typically 0.05, the acceptable probability of a Type I error (rejecting a true H₀).
4. **Collect data and compute a test statistic** — z, t, F, or χ², depending on the situation (see table below).
5. **Compute the p-value** — the probability of seeing data this extreme (or more) if H₀ were actually true.
6. **Decide** — if p < α, reject H₀ in favour of Hₐ; otherwise, fail to reject H₀ (this is *not* the same as proving H₀ true).

> **⚠ A p-value is not the probability H₀ is true**
>
> A p-value of 0.03 means: *if H₀ were true, there's a 3% chance of seeing data this extreme by pure sampling variation.* It says nothing directly about the probability that H₀ is true or false — a distinction that trips up even experienced practitioners, and one worth re-reading before you hit the Expected Value framework on the Evaluation & Validation page.

## Which Test? A Quick Reference

| Test | Use When | Example |
|---|---|---|
| z-test | Population σ known, or n ≥ 30 | Is average petrol price ≠ known population mean? |
| t-test | Population σ unknown, small n | Do two store locations have different average basket sizes? |
| F-test / ANOVA | Comparing means across 3+ groups | Do 4 Zomato delivery zones have different average delivery times? |
| Chi-square | Association between two categorical variables | Is payment method (UPI/Card/Cash) associated with city? |
| Chi-square Goodness-of-Fit | Does one categorical variable's observed distribution match an expected/theoretical one? | Do this month's UPI/Card/Cash shares match last year's reported 60/25/15 split? |
| Correlation (Pearson/Spearman) | Strength & direction of relationship, two continuous variables | Does ad spend correlate with monthly sales? |

## Worked Example — z-test

An Ahmedabad fuel-retail association claims the mean petrol price across its member stations is ₹96.00/litre with a known population standard deviation of σ = ₹2.10/litre. A sample of 50 stations gives a mean of ₹96.85/litre. Is this significantly different from the claimed ₹96.00?

$$\begin{gathered}H_0: \mu = 96.00 \qquad H_a: \mu \neq 96.00 \qquad (\text{two-tailed}, \alpha=0.05)\\[6pt]z = \frac{\bar{x}-\mu}{\sigma/\sqrt{n}} = \frac{96.85-96.00}{2.10/\sqrt{50}} = 2.86\end{gathered}$$

At α = 0.05 (two-tailed), the critical z-values are ±1.96. Since |2.86| > 1.96, we **reject H₀**: the sample mean is statistically significantly different from ₹96.00/litre — this is not just sampling noise.

> **📋 Where does ±1.96 come from? Use a z-table**
>
> The critical value ±1.96 is read off a **standard normal (z) table**, which gives the cumulative area under the bell curve to the left of any z-score. For a two-tailed test at α = 0.05, you want the z beyond which only 2.5% of the area remains in each tail — i.e. the z where the cumulative area is 0.975 — which the table gives as 1.96. Two good free references: the interactive table at [ztable.net](https://www.ztable.net/), and a printable university PDF from the [University of Arizona](https://math.arizona.edu/~jwatkins/normal-table.pdf). (In Python, `scipy.stats.norm.ppf(0.975)` returns 1.9599… directly — no table lookup needed.)

> **💡 Sample size matters**
>
> Repeat the same calculation with only n = 10 stations instead of 50: z = (0.85)/(2.10/√10) = 1.28, which falls *inside* ±1.96 — so with the smaller sample, you would fail to reject H₀, even though the underlying difference in price is identical. Larger samples shrink the standard error and make real effects easier to detect — exactly why sample-size planning (the Minimum Detectable Effect framework) gets its own worked example in the A/B Testing full project.

## Worked Example — t-test (two independent samples)

A Surat retailer wants to know whether two of its store locations have different average basket sizes (₹). Population σ is unknown and samples are small, so a t-test (not a z-test) applies. Store A: 12 baskets, mean ₹555.42. Store B: 10 baskets, mean ₹662.00.

$$\begin{gathered}H_0: \mu_A = \mu_B \qquad H_a: \mu_A \neq \mu_B \qquad (\text{two-tailed}, \alpha=0.05)\\[6pt]t = \frac{\bar{x}_A-\bar{x}_B}{SE_{\text{pooled}}} = -5.231, \qquad df = n_A+n_B-2 = 20\end{gathered}$$

At α = 0.05 (two-tailed) with df = 20, the critical t-values are ±2.086 (read from a *t*-table, the small-sample cousin of the z-table). Since |−5.231| > 2.086 — equivalently, the two-tailed p-value is 0.00004 — we **reject H₀**: the two stores' average basket sizes differ significantly. In Python: `scipy.stats.ttest_ind(A, B)` returns exactly this t and p.

## Worked Example — F-test / ANOVA (3+ groups)

Do four Zomato delivery zones have different average delivery times? Running three separate t-tests between pairs would inflate the false-positive rate (the multiple-comparisons problem), so ANOVA tests all four group means simultaneously in one F-test. Zone means (minutes): 30.5, 39.5, 26.5, 35.5 (6 observations each).

$$\begin{gathered}H_0: \mu_1=\mu_2=\mu_3=\mu_4 \qquad H_a: \text{at least one differs} \qquad (\alpha=0.05)\\[6pt]F = \frac{\text{variance between groups}}{\text{variance within groups}} = 41.28, \qquad df = (3,20)\end{gathered}$$

The critical F-value at α = 0.05 with df (3, 20) is 3.098. Since 41.28 > 3.098 (p < 0.00001), we **reject H₀**: the zones do have significantly different average delivery times. ANOVA only tells you *that* a difference exists — a follow-up pairwise test (with a Bonferroni correction) identifies *which* zones differ. In Python: `scipy.stats.f_oneway(z1, z2, z3, z4)`.

## Worked Example — Chi-square test of association

Is payment method (UPI / Card / Cash) associated with city? This is a relationship between two *categorical* variables, so a chi-square test of independence applies — comparing the observed counts in a contingency table against the counts you'd expect if the two variables were completely independent.

$$\begin{gathered}H_0: \text{payment method and city are independent} \qquad H_a: \text{they are associated}\\[6pt]\chi^2 = \sum\frac{(\text{Observed}-\text{Expected})^2}{\text{Expected}} = 11.472, \qquad dof=(rows-1)(cols-1)=4\end{gathered}$$

The critical χ² at α = 0.05 with 4 degrees of freedom is 9.488. Since 11.472 > 9.488 (p = 0.022), we **reject H₀**: payment-method preference is significantly associated with city — the payment mix genuinely differs from one city to another rather than being the same everywhere. In Python: `scipy.stats.chi2_contingency(table)`.

## Worked Example — Chi-square goodness-of-fit

A different chi-square question: does *one* categorical variable's observed distribution match an expected theoretical split? This month's 1,000 transactions were 540 UPI / 290 Card / 170 Cash. Does that match last year's reported 60% / 25% / 15% mix?

$$\begin{gathered}H_0: \text{this month matches the 60/25/15 split} \qquad H_a: \text{it does not}\\[6pt]\text{Expected: } 600/250/150. \qquad \chi^2=\sum\frac{(O-E)^2}{E}=15.067, \qquad dof=\text{categories}-1=2\end{gathered}$$

The critical χ² at α = 0.05 with 2 degrees of freedom is 5.991. Since 15.067 > 5.991 (p = 0.0005), we **reject H₀**: this month's payment mix has shifted significantly from last year's split — UPI is up and Cash is down by more than sampling noise would explain. In Python: `scipy.stats.chisquare(observed, f_exp=expected)`.

## Worked Example — Correlation significance

Does ad spend correlate with monthly sales? Across 10 months of data (both in ₹ lakh), Pearson's r measures the strength and direction of the linear relationship — and a t-test on r tells us whether that correlation is significantly different from zero, or could just be coincidence in a small sample.

$$\begin{gathered}H_0: \rho = 0\ (\text{no linear relationship}) \qquad H_a: \rho \neq 0\\[6pt]r = 0.9941, \qquad t = r\sqrt{\dfrac{n-2}{1-r^2}} = 25.90, \qquad df = n-2 = 8\end{gathered}$$

The critical t at α = 0.05 (two-tailed) with df = 8 is ±2.306. Since 25.90 > 2.306 (p < 0.000001), we **reject H₀**: ad spend and sales are very strongly and significantly positively correlated. But remember the two cautions below — this significant r confirms a strong *linear association*, not that ad spend *causes* the sales. In Python: `scipy.stats.pearsonr(ad, sales)`.

## Confidence Intervals

A confidence interval gives a range of plausible values for a population parameter, rather than a single point estimate.

$$\text{CI} = \text{Point Estimate} \pm (\text{Critical Value} \times \text{Standard Error})$$

A 95% confidence interval doesn't mean "95% probability the true mean is in this range" — it means that if you repeated the sampling process many times, about 95% of the intervals constructed this way would contain the true population mean. This exact machinery reappears, computed via resampling instead of a formula, on the **Bootstrap Resampling & CI** page.

## Correlation

$$r = \frac{\sum (X-\bar{X})(Y-\bar{Y})}{\sqrt{\sum (X-\bar{X})^2 \cdot \sum (Y-\bar{Y})^2}}$$

r ranges from −1 (perfect negative) to +1 (perfect positive), with 0 meaning no linear relationship. Two cautions worth internalising early: **correlation is not causation**, and Pearson's r only captures *linear* relationships — a strong curved (e.g. U-shaped) relationship can produce an r close to zero despite being highly predictable. Spearman's rank correlation is the non-parametric alternative, used when one variable is ordinal rather than continuous.

## Permutation Test — A Distribution-Free Alternative

The t-test above assumes the underlying data is (at least approximately) normally distributed. When that assumption is shaky — small samples, or a clearly skewed distribution — a **permutation test** sidesteps it entirely: it repeatedly shuffles the group labels on the actual observed data, recomputes the test statistic (e.g. difference of means) for every possible (or a large random sample of) relabelling, and asks how extreme the real, unshuffled result is relative to that shuffled distribution.

> **💡 Real numbers: petting vs. praise (Agresti & Kateri)**
>
> Comparing two small samples (n=7 each) of dog calming-time scores under "petting" versus "praise" conditions, a permutation test using the difference of means as the test statistic gave a one-sided P-value of 0.0035 — versus a classical one-sided t-test P-value of 0.0017 on the same data. Both point to the same conclusion (petting produces a significantly larger effect), but the permutation test reaches it without assuming the underlying calming-time distribution is normal — valuable precisely because n=7 is too small to check that assumption reliably in the first place.

The tradeoff: permutation tests are computationally heavier (often thousands of reshufflings) and are typically reserved for cases where sample size is too small, or the distribution too skewed, to trust a parametric test's normality assumption.

## Practical Stats in Python: SciPy & StatsModels

Two libraries do the heavy lifting for statistical analysis. **SciPy** (`scipy.stats`) provides tests and distributions; **StatsModels** provides rich statistical *models* — most usefully, ordinary least-squares regression with a full inferential summary.

### Reading an OLS Regression Summary

Fitting a line is easy; the value is in the *inference* StatsModels reports around it. We fit `Y ≈ β₀ + β₁X` on 50 points and read the key numbers.

In [ ]:
import numpy as np
import statsmodels.api as sm

np.random.seed(42)
X = np.random.rand(50, 1) * 10
Y = 2 * X.ravel() + np.random.randn(50) * 2 + 1

X_c = sm.add_constant(X)          # add the intercept column
model = sm.OLS(Y, X_c).fit()

print("R-squared:", round(model.rsquared, 3))
print("intercept (const):", round(model.params[0], 4))
print("slope (x1):", round(model.params[1], 4))
print("x1 t-statistic:", round(model.tvalues[1], 3))
print("x1 p-value:", round(model.pvalues[1], 6))
print("x1 95% CI:", [round(v, 3) for v in model.conf_int()[1]])

| Quantity | What it tells you |
|---|---|
| **R-squared** | Share of variance in Y explained by the model (0.905 → 90.5%). |
| **coef** | The estimated intercept and slope (Y rises ≈1.96 per unit X). |
| **std err** | Uncertainty in each coefficient — how much it would vary across samples. |
| **t = coef / std err** | How many standard errors the coefficient sits from zero. |
| **P>\|t\|** | p-value: probability of a t this large if the true coefficient were 0. Here ≈0 → highly significant. |
| **[0.025, 0.975]** | 95% confidence interval for the coefficient; excludes 0 → significant. |

The red line is the fitted model; the green band is the 95% confidence interval for the mean response.

## Multicollinearity

When two predictors are strongly correlated, a regression can't tell their effects apart — coefficients become unstable and hard to interpret. The **Variance Inflation Factor (VIF)** flags it: VIF > 10 signals serious multicollinearity.

In [ ]:
import numpy as np, pandas as pd
from statsmodels.stats.outliers_influence import variance_inflation_factor

np.random.seed(7)
x1 = np.random.rand(100)
x2 = x1 * 0.95 + np.random.rand(100) * 0.05   # nearly a copy of x1
x3 = np.random.rand(100)                       # independent
X = pd.DataFrame({'x1': x1, 'x2': x2, 'x3': x3})

for i, col in enumerate(X.columns):
    print(f"VIF {col}: {variance_inflation_factor(X.values, i):.2f}")

The sky-high VIFs for `x1` and `x2` confirm they carry the same information; the fix is to drop one, or combine them. `x3`, being independent, is fine.

## Non-Parametric Tests

The z-, t-, and F-tests assume roughly normal data. When that fails — small samples, ordinal data, heavy skew — use **non-parametric** tests, which work on *ranks* instead of raw values and assume far less.

| Non-parametric test | Replaces | Compares |
|---|---|---|
| Mann–Whitney U | Two-sample t-test | Two independent groups |
| Kruskal–Wallis | One-way ANOVA | Three or more groups |
| Spearman's ρ | Pearson correlation | Monotonic association (via ranks) |

In [ ]:
import numpy as np
from scipy import stats

np.random.seed(3)
# Mann-Whitney U: two independent groups
A = np.random.normal(20, 5, 30)
B = np.random.normal(24, 5, 30)
u, p = stats.mannwhitneyu(A, B)
print(f"Mann-Whitney U={u:.1f}, p={p:.4f}")

# Kruskal-Wallis: three groups
g1 = np.random.normal(20, 5, 25)
g2 = np.random.normal(23, 5, 25)
g3 = np.random.normal(26, 5, 25)
h, p = stats.kruskal(g1, g2, g3)
print(f"Kruskal-Wallis H={h:.3f}, p={p:.4f}")

# Spearman: monotonic (non-linear) relationship
x = np.linspace(0, 10, 100)
y = x**2 + np.random.normal(0, 3, 100)      # curved, but monotonic
rho, p = stats.spearmanr(x, y)
print(f"Spearman rho={rho:.4f}")

> **💡 Reading the results**
>
> All three p-values here are below 0.05, so each test rejects its null hypothesis: the groups differ (Mann–Whitney, Kruskal–Wallis) and x & y are strongly associated (Spearman ρ ≈ 0.99). Spearman catches the curved relationship that Pearson's linear correlation would understate.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · A one-sample t-test

Fernwood Media claims its average watch time is 50 minutes. Test that claim on the sample below with `stats.ttest_1samp`. Store the p-value in `p` and `reject` = `True` if p < 0.05.

In [ ]:
from scipy import stats
minutes = [44, 47, 52, 41, 45, 43, 48, 46, 42, 49, 45, 44]
p = reject = None   # TODO


In [ ]:
try:
    check("p-value is small", p < 0.01)
    check("reject", reject is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from scipy import stats
minutes = [44, 47, 52, 41, 45, 43, 48, 46, 42, 49, 45, 44]
p = stats.ttest_1samp(minutes, 50).pvalue
reject = bool(p < 0.05)

```

</details>

### Exercise 2 · Medium · A/B test with Welch's t-test

Two checkout pages were tried. Compare `basket_a` and `basket_b` with a **Welch** two-sample t-test (`equal_var=False`). Store the p-value in `p_ab` and the difference of means `B - A` in `lift`.

In [ ]:
from scipy import stats
import numpy as np
rng = np.random.default_rng(11)
basket_a = rng.normal(500, 80, 120)
basket_b = rng.normal(535, 95, 130)
p_ab = lift = None   # TODO


In [ ]:
try:
    check("lift is positive", lift > 0)
    check("difference is significant", p_ab < 0.05)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
from scipy import stats
import numpy as np
rng = np.random.default_rng(11)
basket_a = rng.normal(500, 80, 120)
basket_b = rng.normal(535, 95, 130)
p_ab = stats.ttest_ind(basket_b, basket_a, equal_var=False).pvalue
lift = basket_b.mean() - basket_a.mean()

```

</details>

### Exercise 3 · Stretch · A confidence interval by hand

Compute the 95% confidence interval for the mean of `x` using the t distribution: `mean ± t* × SEM` with `stats.t.ppf`. Store it as `(lo, hi)` in `ci` and check it against `stats.t.interval`.

In [ ]:
import numpy as np
from scipy import stats
x = np.array([12.1, 11.8, 12.6, 12.3, 11.9, 12.4, 12.0, 12.2])
ci = None   # TODO


In [ ]:
try:
    ref = stats.t.interval(0.95, df=len(x) - 1, loc=x.mean(), scale=stats.sem(x))
    check("matches scipy", np.allclose(ci, ref))
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
import numpy as np
from scipy import stats
x = np.array([12.1, 11.8, 12.6, 12.3, 11.9, 12.4, 12.0, 12.2])
sem = x.std(ddof=1) / np.sqrt(len(x))
tstar = stats.t.ppf(0.975, df=len(x) - 1)
ci = (x.mean() - tstar * sem, x.mean() + tstar * sem)

```

</details>

---
*Back to the course: **Machine Learning End To End → Inferential Statistics**.*